## Import the Libraries

In [ ]:
import time
import re
import wikipediaapi
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph
import json
import os
from neo4j import GraphDatabase
from langchain_huggingface import HuggingFaceEmbeddings

### API Keys - Neo4J and Groq

In [ ]:
GROQ_API_KEY = "XXXXX"
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "XXXX"

### Initial set of Wikipedia Articles

* We start with a small set of wikipedia articles on important US and International Legislative entities.
* We recursively iterate through each of these articles, and extract the articles they directly link to.

For example, if we're inside the wikipedia article `Supreme Court of the United States` and this particular article references the article `Constitution of the United States`, then we extract the contents and sections from the referenced article as well. We use a Iterative Deepening Depth First Search approach, but due to limited computation, we set the search depth to 1, we do not go further into the referenced articles.

In [ ]:
WIKI_TITLES = [
    "Supreme Court of the United States",
    "Constitution of the United States",
    "Geneva Conventions",
    "Universal Declaration of Human Rights",
    "International Criminal Court",
    "Brown v. Board of Education",
    "Fourth Amendment to the United States Constitution"
]

We utilize the wikipedia api to make things easier, where we can directly extract page contents, summary and title via a wikipedia article.

In [12]:
wiki = wikipediaapi.Wikipedia(user_agent="aayush_wiki",language="en")

### Scrape Wikipedia Articles

We define -
- a visited set, to keep track of the articles
- articles dictionary with the format {'title': {"text": text, "citations": citations}}
- Set the depth to 1


1. We pass the list of base wikipedia articles to the `fetch_articles` function
2. In the function, we check if the title is already visited, or if we're at a greater depth
3. We make the article visited, if the article is already present in the cached directory, we load it directly
4. If not, we fetch the wikipedia article from the api using wiki.page(title)
5. If the page exists, we get all the summary, section contents, and any reference URLs
6. In recurse_sections() we collect the text and any reference URLs from each subsections
7. The combined article data (text + citations) is saved to the article dictionary
8. We can find every single linked article of the page using page.links.keys(), we recursively call fetch_articles() on the linked titles, and increase the current_depth by 1 (DFS)

In [ ]:
visited = set()
articles = {}
depth = 1

def recurse_sections(sections):
    content = ""
    references = []
    for section in sections:
        content += f"\n\n== {section.title} ==\n{section.text}"
        contents, refs = recurse_sections(section.sections)
        content += contents
        references.extend(refs)
        if "reference" in section.title.lower():
            urls = re.findall(r'https?://\S+', section.text)
            references.extend(urls)
    return content, references
    
def get_full_sections_with_citations(title):
    page = wiki.page(title)
    content, references = recurse_sections(page.sections)
    text = page.summary + content
    return text, list(set(references))

def fetch_articles(titles, current_depth=0, depth=1):
    for title in titles:
        if title in visited or current_depth > depth:
            return
        
        visited.add(title)
        cache_path = f"cache/{title.replace('/', '_').replace(' ', '_')}.json"
        
        if os.path.exists(cache_path):
            print(f"Loaded from cache: {title}")
            with open(cache_path, "r", encoding="utf-8") as f:
                articles[title] = json.load(f)
        else:
            page = wiki.page(title)
            if not page.exists():
                print(f"Page not found: {title}")
                return

            text, citations = get_full_sections_with_citations(title)
            articles[title] = {"text": text, "citations": citations}
            
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(articles[title], f, ensure_ascii=False, indent=2)

            time.sleep(1.5)

        for linked_page in page.links.keys():
            fetch_articles(linked_page, current_depth + 1)
    
    return articles

### Build the Graph

For this we use a LLM to help transform unstructured legal text into a structured knowledge graph, we initialize an LLM using the ChatGroq class from Langchain, which allows us to utilize a hosted `llama3-70b-8192` model via the Groq API.

We configure an LLMGraphTransformer object from Langchain that takes the LLM as input and uses it to generate graph nodes and relationships from the processed text.

In [ ]:
llm = ChatGroq(groq_api_key=GROQ_API_KEY, model="llama3-70b-8192")

graph_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Law", "Court", "Case", "Person", "Institution", "Treaty", "Amendment", "Document", "Country", "Government", "Article", "Organization"],
    allowed_relationships=["decided_by", "ratified_by", "interprets", "applies_to", "established_by", "includes", "part_of", "describes", "has", "superseded"],
    node_properties=True,
    relationship_properties=True,
    strict_mode=False
)

Fetch the articles from Wikipedia

In [ ]:
print("Fetching Wikipedia Articles")
articles = fetch_articles(WIKI_TITLES, current_depth = 0, depth = 1)

Convert the articles dictionary to the documents object from Langchain, due to the Llama model's 8000 token constraints, we limit the text length to 4000.

In [ ]:
documents = []
for title, data in articles.items():
    documents.append(Document(
        page_content=data["text"][:4000],
        metadata={
            "source": f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}",
            "citations": data.get("citations", [])
        }
    ))

#### Generate and Upload the Graph to Neo4J

In [ ]:
print("Transforming into Graph")
graph_documents = graph_transformer.convert_to_graph_documents(documents)

In [ ]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD
)

graph.add_graph_documents(graph_documents, include_source=True)

### Graph Structure

![Alt text](images/knowledge_graph_image.png)

## Embed the Graph

We generate and store vector embeddings for text nodes in the graph, using the sentence-transformers/all-MiniLM-L6-v2 embedder. We retreive all Document nodes that have a text property but no embedding, and convert each retreived document to it's particular dense vector embedding form to capture semantic meaning of the text, which would be later used in the RAG pipeline to retreive additional "Knowledge"

In [ ]:
uri = "bolt://localhost:7687"
driver = GraphDatabase.driver(uri, auth=("neo4j", "XXXXX"))
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/Users/aayushsubramaniam/Documents/University at Buffalo/Spring 2025/Deep Learning/Final Project Submission/env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def embed_and_store(tx, external_id, vector):
    tx.run(
        """
        MATCH (d:Document) WHERE d.external_id = $external_id
        SET d.embedding = $vector
        """,
        external_id=external_id,
        vector=vector
    )

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (d:Document)
        WHERE d.embedding IS NULL AND d.text IS NOT NULL
        RETURN d.external_id AS ext_id, d.text AS text
        LIMIT 500
    """)

    for record in result:
        external_id = record["ext_id"]
        text = record["text"]
        vector = embedder.embed_query(text)
        session.execute_write(embed_and_store, external_id, vector)

print("Embeddings stored in Neo4j!")

### References

- Wikipedia API : https://wikipedia-api.readthedocs.io/en/latest/
-  https://github.com/martin-majlis/Wikipedia-API#sections-and-subsections
- Iterative Deepening  to get nested Articles : https://en.wikipedia.org/wiki/Iterative_deepening_depth-first_search
- ChatGroq : https://python.langchain.com/docs/integrations/chat/groq/
- https://console.groq.com/docs/quickstart
- Basic Graph implementation with Neo4j : https://python.langchain.com/docs/tutorials/graph/
- LLM Graph Transformer : https://python.langchain.com/api_reference/_modules/langchain_experimental/graph_transformers/llm.html#create_unstructured_prompt
- https://medium.com/data-science/building-knowledge-graphs-with-llm-graph-transformer-a91045c49b59
- Neo4j Integration : https://neo4j.com/labs/genai-ecosystem/langchain/?utm_source=GSearch&utm_medium=PaidSearch&utm_campaign=Evergreen&utm_content=AMS-Search-SEMCE-DSA-None-SEM-SEM-NonABM&utm_term=&utm_adgroup=DSA-GenAI&gad_source=1&gad_campaignid=21955414262&gbraid=0AAAAADk9OYpspbZCwBcBpFeDOdMWl306K&gclid=Cj0KCQjwt8zABhDKARIsAHXuD7Z_I9s5qxxUzpq1aghIQBe3s
- https://medium.com/@la_boukouffallah/how-to-build-a-knowledge-graph-using-neo4j-and-langchain-d2b13dbaf9b8
- Storing the extracted documents : https://python.langchain.com/api_reference/core/documents/langchain_core.documents.base.Document.html
- Vector Index : https://neo4j.com/docs/cypher-manual/current/indexes/semantic-indexes/vector-indexes/